# Monod or Contois? Designing the fed-batch run that tells two growth laws apart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/05_design_discrimination/a_hunter_reiner/tutorial.ipynb)

**Model-based design for model discrimination** (`mbdoe_md`, Hunter-Reiner criterion).

Two growth laws fit the preliminary data equally well:

* **MI, Monod**: $r = \mu_{max}\, y_2 / (K_s + y_2)$, saturation set by the substrate;
* **MII, Contois**: $r = \mu_{max}\, y_2 / (K_x y_1 + y_2)$, saturation that grows with the biomass.

Their predictions differ only under some operating conditions. `mbdoe_md` finds the run that
maximises the difference between the models' predictions (Hunter-Reiner: the sum of squared
differences, weighted by the measurement noise). One such run usually settles the question.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['MI.py', 'MII.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/05_design_discrimination/a_hunter_reiner/' + f, f)

### The P-test

After each fit MIDDoE gives every candidate a probability from its weighted residuals (chi-square
weights, normalised to 100 %). Similar values mean the data cannot tell the models apart.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.des_md import mbdoe_md

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta = {'MI':  [0.25, 0.25, 0.88, 0.09],            # Monod + maintenance (the 'real' process here)
         'MII': [0.24987, 0.00912, 0.94194, 0.10154]}  # Contois: mu_max, Kx, Yxs, m
models = {
    'can_m': ['MI', 'MII'],
    'krt': {m: 'pys' for m in theta},
    'src': {m: str(HERE / f'{m}.py') for m in theta},
    'theta': {m: [x * 1.2 for x in v] for m, v in theta.items()},
    't_l': {m: [x * 0.5 for x in v] for m, v in theta.items()},
    't_u': {m: [x * 1.5 for x in v] for m, v in theta.items()},
}
insilicos = {'tr_m': 'MI', 'theta': theta['MI'], 'errt': 'rel',   # noise 8 % of the value
             'prels': {'1': {'u1': 0.05, 'u2': 30, 'y10': 5.5}, '2': {'u1': 0.10, 'u2': 30, 'y10': 5.5}}}

system = {
    'tvi': {'u1': {'stps': 5, 'cvp': 'CPF', 'const': 'rel', 'min': 0.05, 'max': 0.2, 'offl': 0.01, 'offt': 0.5},
            'u2': {'stps': 5, 'cvp': 'LPF', 'const': 'rel', 'min': 5, 'max': 35, 'offl': 1, 'offt': 0.5}},
    'tvo': {'y1': {'init': 0, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.08},
            'y2': {'init': 0.01, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.08}},
    'tii': {'y10': {'min': 1, 'max': 5.5}},
    'tio': {},
    't_s': [0, 40], 't_r': 0.4 if QUICK else 0.16, 't_d': 0,
}

iden_opt = {'meth': 'SLSQP', 'ms': False, 'ob': 'WLS', 'maxit': 60 if QUICK else 500, 'tol': 1e-8,
            'var-cov': 'M', 'sens_m': 'central', 'eps': 1e-3,
            'c_plt': False, 'f_plt': True, 'plt_s': True, 'log': False}


def ptest(unc):
    """P-test: the probability (%) that each candidate is the right model, given the data."""
    return pd.Series({m: unc['results'][m]['P'] for m in models['can_m']}, name='P [%]').round(1)

### Two preliminary runs: both models fit

In [ ]:
round_data = {}
expera(system, models, insilicos, {}, expr=1)
expera(system, models, insilicos, {}, expr=2)
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)
save_rounds(1, unc['results'], 'preliminary', round_data, models, iden_opt, unc['obs'], system)
P_before = ptest(unc)
P_before

### Design the discriminating run

In [ ]:
des_opt = {'md_ob': 'HR',          # Hunter-Reiner
           'meth': 'PS', 'eps': 1e-3, 'plt': True,
           'itr': {'maxmd': 15 if QUICK else 100, 'tolmd': 1, 'pps': 30}}
design = mbdoe_md(des_opt, system, models, round=2, num_parallel_runs=1 if QUICK else 4)
print('HR value of the design:', design['md_obj'])

The designed run starts with little biomass and moves the feed so that the biomass
concentration changes a lot: that is where Contois (saturation proportional to biomass) and Monod
disagree.

### Run it and test again

In [ ]:
expera(system, models, insilicos, design, expr=3, swps=design['swps'])
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)
save_rounds(2, unc['results'], 'MBDoE-MD (HR)', round_data, models, iden_opt, unc['obs'], system)
pd.concat({'2 preliminary runs': P_before, '+ 1 designed run': ptest(unc)}, axis=1)

**Try:** compare with a third run chosen by hand (for example
`insilicos['prels']['3'] = {'u1': 0.15, 'u2': 20, 'y10': 2.0}` and `expera(..., {}, expr=3)`): it
separates the models much less.